In [ ]:
import os

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
from dotenv import find_dotenv, load_dotenv
from great_tables import GT

conn = duckdb.connect()

load_dotenv(find_dotenv())


True

In [2]:
MINIO_ACCESS_KEY = os.getenv("MINIO_ROOT_USER")
MINIO_SECRET_KEY = os.getenv("MINIO_ROOT_PASSWORD")
MINIO_ENDPOINT_DUCKDB = os.getenv("MINIO_ENDPOINT_DUCKDB")
# Install and load spatial extension
conn.execute("INSTALL spatial; LOAD spatial;")

# for S3/MinIO access
conn.execute("INSTALL httpfs")
conn.execute("LOAD httpfs")

#  MinIO connection
conn.execute("SET s3_region = 'us-east-1'")  # Region doesn't matter here
conn.execute(f"SET s3_endpoint = '{MINIO_ENDPOINT_DUCKDB}'")
conn.execute(f"SET s3_access_key_id = '{MINIO_ACCESS_KEY}'")
conn.execute(f"SET s3_secret_access_key = '{MINIO_SECRET_KEY}'")
conn.execute("SET s3_use_ssl = false")
conn.execute("SET s3_url_style = 'path'")  # Important for MinIO!

In [3]:
# Set up matplotlib style
plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "#f0f0f0"
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.size"] = 10

In [17]:
result = conn.execute("""
    SELECT
    c.name,
    p.year,
    p.population
FROM read_parquet('s3://lakehouse-processed/dim_communes_france.parquet') AS c
JOIN read_parquet('s3://lakehouse-processed/fact_population.parquet') AS p
    ON c.id = p.id
WHERE c.id='09160'
ORDER BY year DESC
LIMIT 5
""").df()
print(result)

        name  year  population
0  Lavelanet  2023        5987
1  Lavelanet  2022        6018
2  Lavelanet  2021        6035
3  Lavelanet  2020        6052
4  Lavelanet  2019        6031


In [79]:
result = conn.execute("""
    SELECT
    *
FROM read_parquet('s3://lakehouse-processed/dim_communes_france.parquet') AS c
JOIN read_parquet('s3://lakehouse-processed/fact_population.parquet') AS p
    ON c.id = p.id
WHERE c.name='Paris'
AND P.year >=2007
ORDER BY p.year DESC
""").df()
result

,id,current_code,name,name_upper,name_lower,siren_code,arrondissement_name,arrondissement_code,department_name,department_code,...,centroid,bbox_xmin,bbox_xmax,bbox_ymin,bbox_ymax,perimeter,number_enclaves,id_1,year,population
0,75056,75056,Paris,PARIS,paris,217500016,Paris,751,Paris,75,...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",2.224219,2.469834,48.815562,48.902148,0.647777,0,75056,2023,2103778
1,75056,75056,Paris,PARIS,paris,217500016,Paris,751,Paris,75,...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",2.224219,2.469834,48.815562,48.902148,0.647777,0,75056,2022,2113705
2,75056,75056,Paris,PARIS,paris,217500016,Paris,751,Paris,75,...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",2.224219,2.469834,48.815562,48.902148,0.647777,0,75056,2021,2133111
3,75056,75056,Paris,PARIS,paris,217500016,Paris,751,Paris,75,...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",2.224219,2.469834,48.815562,48.902148,0.647777,0,75056,2020,2145906
4,75056,75056,Paris,PARIS,paris,217500016,Paris,751,Paris,75,...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",2.224219,2.469834,48.815562,48.902148,0.647777,0,75056,2019,2165423
5,75056,75056,Paris,PARIS,paris,217500016,Paris,751,Paris,75,...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",2.224219,2.469834,48.815562,48.902148,0.647777,0,75056,2018,2175601
6,75056,75056,Paris,PARIS,paris,217500016,Paris,751,Paris,75,...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",2.224219,2.469834,48.815562,48.902148,0.647777,0,75056,2017,2187526
7,75056,75056,Paris,PARIS,paris,217500016,Paris,751,Paris,75,...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",2.224219,2.469834,48.815562,48.902148,0.647777,0,75056,2016,2190327
8,75056,75056,Paris,PARIS,paris,217500016,Paris,751,Paris,75,...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",2.224219,2.469834,48.815562,48.902148,0.647777,0,75056,2015,2206488
9,75056,75056,Paris,PARIS,paris,217500016,Paris,751,Paris,75,...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",2.224219,2.469834,48.815562,48.902148,0.647777,0,75056,2014,2220445


In [66]:
def get_commune_population(commune_id: str, conn) -> pd.DataFrame:
    """
    Query population data for a specific commune by ID.

    Args:
        commune_id: The commune identifier (string)
        conn: Optional DuckDB connection. If None, creates a new one.

    Returns:
        DataFrame with name, year, population columns sorted by year descending
    """

    query = """
    SELECT
        c.name,
        c.department_name,
        c.department_code,
        p.year,
        p.population,
        p2.population AS previous_population,
        p.population - previous_population AS 'Population Evolution',
        ROUND((p.population - p2.population) * 100.0 / p2.population, 2) AS "Population Evolution (%)"
    FROM read_parquet('s3://lakehouse-processed/dim_communes_france.parquet') AS c
    JOIN read_parquet('s3://lakehouse-processed/fact_population.parquet') AS p
        ON c.id = p.id
    LEFT JOIN  read_parquet('s3://lakehouse-processed/fact_population.parquet') AS p2
        ON c.id = p2.id
        AND p.id = p2.id
        AND p.year-1=p2.year
    WHERE c.id=?
    AND P.year >=2007
    ORDER BY p.year DESC
    """

    df = conn.execute(query, [commune_id]).df()

    if df.empty:
        print(f"Warning: No data found for commune ID: {commune_id}")

    return df

In [80]:
# Generate report for Lavelanet
commune_id = "75056"

In [81]:
df = get_commune_population(commune_id, conn)
df.head()


,name,department_name,department_code,year,population,previous_population,Population Evolution,Population Evolution (%)
0,Paris,Paris,75,2023,2103778,2113705,-9927,-0.47
1,Paris,Paris,75,2022,2113705,2133111,-19406,-0.91
2,Paris,Paris,75,2021,2133111,2145906,-12795,-0.60
3,Paris,Paris,75,2020,2145906,2165423,-19517,-0.90
4,Paris,Paris,75,2019,2165423,2175601,-10178,-0.47


In [ ]:
def create_population_table(df: pd.DataFrame, title: str = None) -> GT:
    if df.empty:
        return None

    commune_name = df["name"].iloc[0]
    department_name = df["department_name"].iloc[0]
    department_code = df["department_code"].iloc[0]
    max_year = df["year"].iloc[0]
    min_year = df["year"].iloc[-1]

    table_df = df[
        [
            "year",
            "population",
            "Population Evolution",
            "Population Evolution (%)",
        ]
    ].copy()

    # Add colored arrows to Population Evolution
    def _fmt_arrow(v, fmt: str, suffix: str = "") -> str:
        if pd.isna(v):
            return "—"
        if v > 0:
            return f'<span style="color:green">&#9650; {v:{fmt}}{suffix}</span>'
        if v < 0:
            return f'<span style="color:red">&#9660; {v:{fmt}}{suffix}</span>'
        return f'<span style="color:gray">&#9644; {v:{fmt}}{suffix}</span>'

    table_df["Population Evolution"] = table_df["Population Evolution"].apply(
        lambda v: _fmt_arrow(v, "+,.0f")
    )
    table_df["Population Evolution (%)"] = table_df["Population Evolution (%)"].apply(
        lambda v: _fmt_arrow(v, "+.1f", suffix="%")
    )

    return (
        GT(table_df)
        .tab_header(
            title=f"Population for {commune_name} ({department_name} - {department_code})",
            subtitle=f"Data for Years {min_year} - {max_year}",
        )
        .fmt_integer(columns="population")
        .fmt_markdown(columns="Population Evolution")
        .fmt_markdown(columns="Population Evolution (%)")
        .cols_align(
            align="right", columns=["Population Evolution", "Population Evolution (%)"]
        )
    )

In [95]:
gt_table = create_population_table(df)
gt_table.save(f"{commune_id}.png")
gt_table.show()

Population for Paris (Paris - 75) 
 
 
 Data for Years 2007 - 2023 
 
 
 year 
 population 
 Population Evolution 
 Population Evolution (%) 
 
 
 
 
 2023 
 2,103,778 
 ▼ -9,927 
 ▼ -0.5% 
 
 
 2022 
 2,113,705 
 ▼ -19,406 
 ▼ -0.9% 
 
 
 2021 
 2,133,111 
 ▼ -12,795 
 ▼ -0.6% 
 
 
 2020 
 2,145,906 
 ▼ -19,517 
 ▼ -0.9% 
 
 
 2019 
 2,165,423 
 ▼ -10,178 
 ▼ -0.5% 
 
 
 2018 
 2,175,601 
 ▼ -11,925 
 ▼ -0.6% 
 
 
 2017 
 2,187,526 
 ▼ -2,801 
 ▼ -0.1% 
 
 
 2016 
 2,190,327 
 ▼ -16,161 
 ▼ -0.7% 
 
 
 2015 
 2,206,488 
 ▼ -13,957 
 ▼ -0.6% 
 
 
 2014 
 2,220,445 
 ▼ -9,176 
 ▼ -0.4% 
 
 
 2013 
 2,229,621 
 ▼ -11,000 
 ▼ -0.5% 
 
 
 2012 
 2,240,621 
 ▼ -9,354 
 ▼ -0.4% 
 
 
 2011 
 2,249,975 
 ▲ +6,142 
 ▲ +0.3% 
 
 
 2010 
 2,243,833 
 ▲ +9,728 
 ▲ +0.4% 
 
 
 2009 
 2,234,105 
 ▲ +22,808 
 ▲ +1.0% 
 
 
 2008 
 2,211,297 
 ▲ +18,267 
 ▲ +0.8% 
 
 
 2007 
 2,193,030 
 ▲ +11,659 
 ▲ +0.5%